# Chapter 17 &mdash; Lewis Carroll's Babies and Crocodiles, Decided by a Diagram

**Concept 8 of the Chapter 17 decomposition:** *Lewis Carroll's Babies and Crocodiles, Decided by a Diagram*

Three premises, four variables, and the refutation BDD collapses to a single $0$ node &mdash; the picture *is* the proof.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter17-BDD/Concept-Carroll-Babies-And-Crocodiles/Concept-Carroll-Babies-And-Crocodiles.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Bdd            import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Lewis Carroll wrote logic puzzles as a mathematician at play, and they are still the
friendliest way into mechanical reasoning. This is his shortest:

> Babies are illogical.
> Nobody is despised who can manage a crocodile.
> Illogical persons are despised.
>
> **Therefore, babies cannot manage crocodiles.**

Four propositional variables. The puzzle file has shipped in this repository under
`BDD/python/PyBool/examples/` for years with nothing to run it; here it is, unchanged.

**The method.** To show that premises *entail* a conclusion, assert the premises
**together with the negation of the conclusion** and show the result is unsatisfiable.
Every verification tool you will meet works this way, and a BDD reports the answer in
the most direct form there is: the diagram reduces to the single terminal $0$.

That is worth pausing on. A reduced ordered BDD with no path to $1$ is *the constant
false* &mdash; by Concept 6's canonicity, there is only one such diagram. So the
picture is not an illustration of a proof done elsewhere. It is the proof.

One discipline comes with the method, and this notebook applies it: **check that the
premises are satisfiable first.** Premises that contradict each other also give $0$,
and would "prove" the conclusion for entirely the wrong reason.

## 2. Definitions

### The puzzle

In [ ]:
# --- Carroll's puzzle, in the BDD manager's own markup -------------------
# This is BDD/python/PyBool/examples/example_std_files/babies_and_crocs.txt,
# unchanged apart from the comments.  `=>` is implication, `~` negation.
#
#   Babies are illogical.
#   Nobody is despised who can manage a crocodile.
#   Illogical persons are despised.
#   Therefore: babies cannot manage crocodiles.
CROCS = '''
Var_Order : babies illogical despised manageCrocs
P1 = babies => illogical
P2 = illogical => despised
P3 = manageCrocs => ~despised
C  = babies => ~manageCrocs
'''

def spec(main, body=CROCS):
    return body + 'Main_Exp : ' + main + '\n'

<!-- nav-strip -->

---

&larr;&nbsp;[Ch17&nbsp;7.&nbsp;BDD Sizes, Dynamic Reordering, and the NP-Completeness of Ordering](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter17-BDD/Concept-BDD-Sizes-And-Reordering/Concept-BDD-Sizes-And-Reordering.ipynb) &nbsp;&middot;&nbsp; [**Chapter 17** index](https://github.com/ganeshutah/Jove/blob/master/Chapter17-BDD/README.md) &nbsp;&middot;&nbsp; [Ch17&nbsp;9.&nbsp;Carroll's Wise Young Pigs, and Why Every Premise Earns Its Place](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Concept-Notebooks/Chapter17-BDD/Concept-Carroll-Wise-Young-Pigs/Concept-Carroll-Wise-Young-Pigs.ipynb)&nbsp;&rarr;

---

## 3. Tests

**Are the premises consistent?** Ask before proving anything from them.

In [ ]:
prem = bdd(spec('P1 & P2 & P3'))
print('models of the premises :', prem.count)
print('nodes                  :', prem.nodes)
assert prem.count > 0, 'the premises contradict each other -- nothing to prove'

Five worlds satisfy Carroll's premises. Small enough to simply look at.

In [ ]:
cols = ['babies', 'illogical', 'despised', 'manageCrocs']
print('  '.join('%-11s' % c for c in cols))
for m in prem.models:
    print('  '.join('%-11d' % m[c] for c in cols))

The conclusion is already visible. Exactly one of those worlds has a baby in it, and in that world nobody manages a crocodile.

In [ ]:
withbaby = [m for m in prem.models if m['babies'] == 1]
print('worlds containing a baby :', len(withbaby))
for m in withbaby:
    print('   manageCrocs =', m['manageCrocs'])
print()
print("That is the answer, read off five rows.  It does not scale past a")
print("toy, which is why the rest of the notebook does it the other way.")

**The refutation.** Premises, and the conclusion denied.

In [ ]:
ref = bdd(spec('P1 & P2 & P3 & ~C'))
print('satisfying assignments :', ref.count)
print('nodes in the diagram   :', ref.nodes)
print()
print('unsatisfiable, so the premises ENTAIL the conclusion:')
print('   babies cannot manage crocodiles.')
assert ref.count == 0

And here is that diagram. One node.

In [ ]:
ref

Beside the premises on their own, which is what a *satisfiable* formula looks like.

In [ ]:
side_by_side(('the premises  --  %d nodes, %d models' % (prem.nodes, prem.count), prem),
             ('premises AND the denied conclusion  --  %d nodes' % ref.nodes, ref))

**Does every premise earn its place?** Drop each in turn and ask the same question.

In [ ]:
P = ['P1', 'P2', 'P3']
for drop in P:
    kept = [p for p in P if p != drop]
    b = bdd(spec(' & '.join(kept) + ' & ~C'))
    if b.count == 0:
        print('without %s : still proved' % drop)
    else:
        m = b.models[0]
        print('without %s : NOT proved.  Counterexample -- %s'
              % (drop, ', '.join('%s=%d' % (k, m[k]) for k in cols)))

Read the three counterexamples: each is a baby managing a crocodile, and each gets there through the gap the missing premise left.

In [ ]:
print("Every premise is load-bearing.  Carroll built it that way, and you")
print("did not have to take his word for it -- three BDDs settled it.")
print()
print("This is what a proof assistant does for a living, on formulae with")
print("millions of variables instead of four.  The idea does not change:")
print("deny the conclusion, conjoin, and look at what you get.")

## 4. Exercises


1. Change `C` to the *converse*, `manageCrocs => ~babies`. Is it also entailed? Should
   it be?
2. Assert a conclusion that does **not** follow &mdash; `babies => despised` is true
   here, but try `despised => babies`. Read the countermodel.
3. Put the premises in the opposite variable order. The node counts change; the answer
   does not. Which concept of this chapter says that must be so?
4. Add a fourth premise that contradicts the others and re-run the refutation. It
   reports $0$ nodes and "proves" the conclusion. Explain why the satisfiability check
   at the top of this notebook is not optional.
5. Write one of your own syllogisms in the same markup and decide it.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter17-BDD/Concept-Carroll-Babies-And-Crocodiles')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')